# Step 1 – Data Collection for Line Following

Drive the robot manually along the track while capturing labelled images.
Each frame is saved under one of three class folders:
- `dataset/forward/`
- `dataset/left/`
- `dataset/right/`

**How to use:**
1. Run **Cell 1** to start the camera.
2. Run **Cell 2** to display the live feed and the labelling widget.
3. Hold a key in the text box (`w`=forward, `a`=left, `d`=right) — the robot moves **and** an image is saved with that label.
4. Run **Cell 3** (stop) when done. Aim for ~200+ images per class.

In [ ]:
# ── Cell 1: Imports and camera setup ─────────────────────────────────────────
import os, cv2, time, threading
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import pyzed.sl as sl
import motors
from traitlets.config.configurable import SingletonConfigurable
import traitlets

# Create dataset directories
for cls in ['forward', 'left', 'right']:
    os.makedirs(f'dataset/{cls}', exist_ok=True)

robot = motors.MotorsYukon(mecanum=False)

def bgr8_to_jpeg(img):
    return bytes(cv2.imencode('.jpg', img)[1])

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super().__init__()
        self.zed = sl.Camera()
        init = sl.InitParameters()
        init.camera_resolution = sl.RESOLUTION.VGA
        init.depth_mode = sl.DEPTH_MODE.PERFORMANCE
        init.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera open failed:', status)
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        info = self.zed.get_camera_information()
        self.width  = info.camera_configuration.resolution.width
        self.height = info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                raw = self.image.get_data()
                self.color_value = cv2.cvtColor(raw, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

camera = Camera()
camera.start()
print('Camera started. Proceed to Cell 2.')

In [ ]:
# ── Cell 2: Live display + labelled data capture ──────────────────────────────
display_widget = widgets.Image(format='jpeg', width='50%')
count_label    = widgets.Label(value='Saved — forward:0  left:0  right:0')
text_input     = widgets.Text(value='', placeholder='w/a/d', description='Key:', layout=widgets.Layout(width='200px'))
display(widgets.VBox([display_widget, count_label, text_input]))

counts = {'forward': 0, 'left': 0, 'right': 0}

key_to_class = {'w': 'forward', 'a': 'left', 'd': 'right'}
key_to_speed = 0.35

def update_count_label():
    count_label.value = f"Saved — forward:{counts['forward']}  left:{counts['left']}  right:{counts['right']}"

def save_frame(label):
    """Save the current camera frame under the given class folder."""
    frame = camera.color_value.copy()
    # Crop the bottom half only — sky/ceiling is uninformative for line following
    h = frame.shape[0]
    frame_crop = frame[h//2:, :]
    frame_resized = cv2.resize(frame_crop, (224, 112))
    fname = f'dataset/{label}/{label}_{counts[label]:05d}.jpg'
    cv2.imwrite(fname, frame_resized)
    counts[label] += 1
    update_count_label()

def on_key(change):
    val = change['new']
    if not val:
        return
    key = val[-1]
    if key in key_to_class:
        cls = key_to_class[key]
        # Move robot
        if key == 'w':
            robot.forward(key_to_speed)
        elif key == 'a':
            robot.left(key_to_speed)
        elif key == 'd':
            robot.right(key_to_speed)
        time.sleep(0.15)          # short burst
        robot.stop()
        save_frame(cls)           # label = direction used
    else:
        robot.stop()

# Update display in a background thread
def display_loop():
    while camera.thread_runnning_flag:
        if camera.color_value is not None:
            disp = cv2.resize(camera.color_value, None, fx=0.5, fy=0.5)
            # overlay counts
            cv2.putText(disp, f"F:{counts['forward']} L:{counts['left']} R:{counts['right']}",
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            display_widget.value = bgr8_to_jpeg(disp)
        time.sleep(0.05)

disp_thread = threading.Thread(target=display_loop, daemon=True)
disp_thread.start()

text_input.observe(on_key, names='value')
print('Ready. Click the text box and type w/a/d to drive and collect data.')

In [ ]:
# ── Cell 3: Stop camera ───────────────────────────────────────────────────────
camera.stop()
robot.stop()
for cls in ['forward', 'left', 'right']:
    n = len(os.listdir(f'dataset/{cls}'))
    print(f'  {cls}: {n} images')
print('Done. Proceed to Step2_Train_CNN.ipynb')